# Lightningfish - scaling HN to a sample that can support a positive claim

Sample size is the binding constraint on every claim in the findings log. At
n=22-36 against a strong baseline, the binomial test cannot register anything
short of an enormous margin - so runs at that scale can support **negative**
conclusions but not positive ones.

A few hundred stories is impractical on a CPU box and routine on a GPU. This
notebook does that run, with time budgeting and checkpointing so a session
timeout does not cost you everything.

**Rough power intuition:** to clear a 70% baseline at n=40 you need ~85%
accuracy for p<0.05; at n=400 roughly 76% suffices. Scale is what buys the
ability to detect a real but modest edge.

## 1. Setup

Set the sidebar to **Accelerator: GPU T4 x2** (or P100) and **Internet: On**
before running. Internet is required to install Ollama and pull the model.

Do not paste API keys here. This runs a local model and needs none; if you ever
want a Claude-backed run, use Kaggle **Secrets**, never an inline string.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, requests

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("ollama up")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama did not start")

In [ ]:
MODEL = "qwen2.5:7b"

!nvidia-smi --query-gpu=name,memory.total --format=csv
!ollama pull {MODEL}

import requests

# Force a load so /api/ps reports placement. keep_alive=-1 pins the model so it
# is not unloaded between events (an unload/reload cycle mid-run is brutal).
requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL, "prompt": "hi", "stream": False, "keep_alive": -1},
    timeout=600,
)

for m in requests.get("http://localhost:11434/api/ps").json().get("models", []):
    vram = m.get("size_vram", 0)
    print(f"{m['name']}: size_vram={vram / 1e9:.2f} GB")
    assert vram > 0, (
        "model is on CPU, not GPU - check the accelerator is enabled. "
        "Running on CPU here is no faster than a laptop."
    )
print("GPU inference confirmed")

In [ ]:
!git clone --depth 1 https://github.com/rajul-kk/LightningFish.git /kaggle/working/lf
!pip -q install anthropic openai scipy requests pytest

import os, sys

os.chdir("/kaggle/working/lf")
sys.path.insert(0, "/kaggle/working/lf")

# Engine + HN suites only. The finance/service tests pull yfinance, praw, edgar,
# fastapi, modal and psycopg, none of which an HN run touches.
!python -m pytest tests/core tests/hn -q 2>&1 | tail -3

In [ ]:
# Measure real throughput before committing to a long run.
import time

from lightningfish_core.llm_provider import make_provider

provider = make_provider(f"ollama:{MODEL}")
t0 = time.time()
for _ in range(3):
    provider.get_opinion("Output ONLY a number between -1 and 1.", "Rate this: 0.5", f"ollama:{MODEL}")
per_call = (time.time() - t0) / 3
print(f"~{per_call:.2f}s per short LLM call")
print(f"(on a starved CPU box this was ~27s - if you see double digits, check the GPU assert above)")

## 2. Pull a large class-balanced sample

The Algolia API is free and unauthenticated (~10k req/hr), so the pull is cheap;
simulation time is what grows. Seeds are cached, so re-running this cell is free.

In [ ]:
import os

os.environ["LIGHTNINGFISH_MODEL"] = f"ollama:{MODEL}"
os.environ["LIGHTNINGFISH_LOCAL_TIMEOUT"] = "120"

LIMIT = 400          # stories to pull (class-balanced: half high, half low)
N_AGENTS = 24
N_ROUNDS = 4
TIME_BUDGET_MIN = 420   # stop simulating past this and score what we have

from lightningfish_core.event_cache import CachingAdapter, EventCache, cached_pull_events
from lightningfish_hn.backtest_events import pull_hn_events
from lightningfish_hn.config import HNCommentsAdapter, HNDomainAdapter

cache = EventCache("hn_stories")
points_adapter = CachingAdapter(HNDomainAdapter(), cache)
comments_adapter = CachingAdapter(HNCommentsAdapter(), cache)

events = cached_pull_events(
    cache, f"hn:points:{LIMIT}", lambda: pull_hn_events("points", LIMIT)
)
print(f"{len(events)} events available")

## 3. Simulate, with a time budget and checkpoints

A Kaggle session can end before a long run does. This writes a running summary
to `/kaggle/working/progress.json` after every chunk, and stops cleanly once the
time budget is hit rather than being killed mid-run.

In [ ]:
import json, pathlib, time

from lightningfish_core.engine import SimulationEngine

engine = SimulationEngine(points_adapter, model=f"ollama:{MODEL}")

pairs = []
t0 = time.time()
budget_s = TIME_BUDGET_MIN * 60

for i, ev in enumerate(events, 1):
    agents = points_adapter.build_personas(N_AGENTS)
    result = engine.run(ev.seed, agents, n_rounds=N_ROUNDS)
    pairs.append((ev, result))

    if i % 10 == 0 or i == len(events):
        elapsed = time.time() - t0
        rate = elapsed / i
        eta = rate * (len(events) - i)
        print(f"{i}/{len(events)}  {rate:.1f}s/event  "
              f"elapsed {elapsed/60:.1f}m  eta {eta/60:.1f}m", flush=True)
        pathlib.Path("/kaggle/working/progress.json").write_text(json.dumps({
            "done": i,
            "total": len(events),
            "seconds_per_event": rate,
            "finals": [
                (e.event_id, r.trajectory[-1] if r.trajectory else 0.0)
                for e, r in pairs
            ],
        }, indent=2))

    if time.time() - t0 > budget_s:
        print(f"\ntime budget reached at {i}/{len(events)} - scoring what we have")
        break

print(f"simulated {len(pairs)} events in {(time.time()-t0)/60:.1f} minutes")

## 4. Score against the full baseline ladder

In [ ]:
from lightningfish_core.backtest import llm_baseline, score_precomputed, sign


def ladder(adapter):
    return {
        "naive": lambda e: sign(adapter.naive_prediction(e.seed)),
        "single_llm": llm_baseline(adapter, engine),
    }


for label, adapter in (("points / reception", points_adapter),
                       ("num_comments / engagement", comments_adapter)):
    report = score_precomputed(adapter, pairs, baselines=ladder(adapter))
    print(f"\n=== hn {label} ===")
    print(report.summary_line())
    print(f"  n={report.n_events}  sim={report.sim_accuracy:.1%}  "
          f"majority={report.majority_class_accuracy:.1%}")
    for name, acc in report.baseline_accuracy.items():
        beat = "PASS" if report.beats_baselines[name] else "FAIL"
        print(f"  vs {name:<12} {acc:.1%}   {beat}")
    print(f"  p_value_vs_best = {report.p_value_vs_best:.4f}")
    print(f"  parse_rate={report.mean_parse_success_rate:.2f}  "
          f"low_confidence_events={report.low_confidence_events}  "
          f"skipped={report.skipped}")

## 5. Save everything

In [ ]:
!cp -r .cache/lightningfish /kaggle/working/cache
!ls -la /kaggle/working/

## Interpreting a large-n result

With a few hundred events the significance test becomes meaningful, so the
verdict is simply: does the simulation beat **every** rung, and is
`p_value_vs_best` below 0.05?

A negative result at this scale is worth more than a negative at n=36 - it is
the difference between "underpowered" and "genuinely no edge". Either way the
number belongs in the findings log.

One caution: the class-balanced sampler pulls half above the high threshold and
half below the low one, so the majority-class floor sits near 50% by
construction. That makes the naive and single-LLM rungs, not the majority class,
the bars that actually matter.